# Step 1 — Filter OSM PBF by boundary

OSM planet or regional extracts (e.g. from [Geofabrik](https://download.geofabrik.de/)) cover large areas.
This notebook cuts the PBF down to exactly your boundary polygon using **osmium-tool**.

**What it does:**
1. Shows your boundary on an interactive map
2. Calls `osmium extract` to cut the PBF to the boundary polygon
3. Reports the size of the resulting file

**Prerequisites:**
- `osmium-tool` must be installed
  - Ubuntu/Debian: `sudo apt install osmium-tool`
  - macOS: `brew install osmium-tool`
  - Windows: see [osmium-tool docs](https://osmcode.org/osmium-tool/)
- A regional or planet PBF (e.g. `sweden-latest.osm.pbf` from Geofabrik)

**Output:** `pbf/{name}.osm.pbf` — a small PBF containing only the roads inside your boundary

In [ ]:
%%time
import subprocess
from pathlib import Path
import folium
import geopandas as gpd

# ── Configuration — edit these ────────────────────────────────────────────
NAME          = 'sodermalm'                         # used to name output files
PBF_INPUT     = Path('/path/to/sweden-latest.osm.pbf')  # your regional PBF
BOUNDARY_PATH = Path('../boundaries/sodermalm.geojson')  # your boundary
PBF_DIR       = Path('../pbf')                          # output folder for PBFs
# ─────────────────────────────────────────────────────────────────────────

PBF_DIR.mkdir(exist_ok=True)
OUTPUT_PBF = PBF_DIR / f'{NAME}.osm.pbf'

print(f'Input PBF   : {PBF_INPUT}')
print(f'Boundary    : {BOUNDARY_PATH}')
print(f'Output PBF  : {OUTPUT_PBF}')

---
## Boundary preview

Verify the boundary polygon before filtering — a wrong boundary means extracting the wrong area.

In [ ]:
%%time
gdf = gpd.read_file(BOUNDARY_PATH).to_crs('EPSG:4326')
bounds = gdf.total_bounds   # (minx, miny, maxx, maxy)
center = [(bounds[1] + bounds[3]) / 2, (bounds[0] + bounds[2]) / 2]

m = folium.Map(location=center, zoom_start=12, tiles='OpenStreetMap')
folium.GeoJson(
    gdf.__geo_interface__,
    style_function=lambda _: {'color': 'navy', 'weight': 2, 'fillOpacity': 0.15},
    tooltip=NAME,
).add_to(m)
m

---
## Filter PBF

`osmium extract` reads the input PBF and writes a new one containing only features
that intersect the boundary polygon.

The `--overwrite` flag allows re-running without deleting the output manually.

In [ ]:
%%time
if not PBF_INPUT.exists():
    raise FileNotFoundError(f'PBF not found: {PBF_INPUT}\nDownload from https://download.geofabrik.de/')

cmd = [
    'osmium', 'extract',
    '--polygon', str(BOUNDARY_PATH.resolve()),
    '--output',  str(OUTPUT_PBF.resolve()),
    '--overwrite',
    str(PBF_INPUT.resolve()),
]
print('Running:', ' '.join(cmd))
result = subprocess.run(cmd, capture_output=True, text=True)

if result.returncode != 0:
    print('STDERR:', result.stderr)
    raise RuntimeError('osmium extract failed — check osmium-tool is installed')

size_mb = OUTPUT_PBF.stat().st_size / 1_048_576
print(f'\nDone!  {OUTPUT_PBF.name}  ({size_mb:.2f} MB)')
print('Proceed to notebook 2 to build the road network.')